# ITS_LIVE search patterns

This notebook demonstrates the three search backends exposed by `itslive.search(...)` and how to paginate over the results manually.

| `type` | backend | catalog | engine |
|---|---|---|---|
| `"pgstac"` | STAC API (live) | `https://stac.itslive.cloud` | `pystac_client` |
| `"serverless"` | STAC geoparquet | `s3://its-live-data/test-space/stac/catalog/warehouse` | `duckdb` (default) or `rustac` |

The pgstac backend queries the live catalog, while the serverless backends query a partitioned geoparquet snapshot on S3, so result sets may differ slightly for recently ingested data. Both return the same `data` asset hrefs (`.nc` files).

## Setup

`itslive.search(...)` takes a geometry (`bbox`, `polygon`, `geojson`, or a raw `intersects` GeoJSON dict), a date range (`start`/`end`), and backend parameters. Filter helpers (`EQ`, `GTE`, `LTE`, `GT`, `LT`, `NEQ`) are exported from the top-level package.

In [ ]:
import itslive
from itslive import EQ, GTE, LTE, search

# Region of interest: a small bounding box in the Karakoram (Shishper glacier)
bbox = [74.535403, 36.352031, 74.680602, 36.489164]

## 1. STAC API search (`type="pgstac"`)

Queries the live pgstac catalog via `pystac_client`. Returns a sorted, de-duplicated list of data-asset URLs. The `limit` (default 10000) controls how many items are fetched per HTTP request during pagination.

In [ ]:
urls = search(bbox=bbox, start="2022-01-01", end="2022-12-31", type="pgstac")
print(f"{len(urls)} velocity pairs")
print(urls[0] if urls else "(none)")

## 2. Serverless geoparquet search (`type="serverless"`)

Queries the partitioned STAC geoparquet warehouse directly from S3. `engine="duckdb"` is the default; `engine="rustac"` uses rustac's Arrow record-batch path.

In [ ]:
urls_duckdb = search(bbox=bbox, start="2022-01-01", end="2022-12-31",
                    type="serverless", engine="duckdb")
print(f"duckdb: {len(urls_duckdb)} velocity pairs")

In [ ]:
urls_rustac = search(bbox=bbox, start="2022-01-01", end="2022-12-31",
                     type="serverless", engine="rustac")
print(f"rustac: {len(urls_rustac)} velocity pairs")

### Which catalog is more current?

`pgstac` is continuously ingested, while the geoparquet warehouse is a snapshot, so `pgstac` may contain newer items. For fully settled years the sets should match closely:

In [ ]:
print("only in pgstac :", len(set(urls) - set(urls_duckdb)))
print("only in duckdb:", len(set(urls_duckdb) - set(urls)))

## 3. Streaming results (`stream=True`)

For very large result sets (1M+ URLs) pass `stream=True` to get a generator that yields hrefs as they are found, so memory stays flat.

In [ ]:
gen = search(bbox=bbox, start="2022-01-01", end="2022-12-31",
              type="serverless", engine="duckdb", stream=True)
print(type(gen))
print(next(gen))  # first href

## 4. Filters

Friendly parameters (`mission`, `min_interval`/`max_interval`, `percent_valid_pixels`) are translated to STAC property filters, and arbitrary `filters=` override them:

In [ ]:
# mission + time separation + minimum valid pixels
urls = search(bbox=bbox, start="2022-01-01", end="2022-12-31",
              mission="sentinel1", min_interval=12, max_interval=48,
              percent_valid_pixels=50)
print(f"{len(urls)} S1 pairs with 12-48 day separation, >=50% valid pixels")

# arbitrary property filter, e.g. a specific version
urls_v002 = search(bbox=bbox, start="2022-01-01", end="2022-12-31",
                    filters={"version": EQ("002")})
print(f"{len(urls_v002)} version-002 pairs")

## 5. Manual pagination over pgstac results

`itslive.search(...)` hides pagination, but you can page through the raw STAC API yourself with `pystac_client`. Each page is a GeoJSON FeatureCollection; use the `next` link to fetch the next page. This shows how the library paginates under the hood.

In [ ]:
import pystac_client

client = pystac_client.Client.open("https://stac.itslive.cloud")
search = client.search(
    intersects={"type": "Polygon", "coordinates": [[[bbox[0], bbox[1]], [bbox[2], bbox[1]],
                                                          [bbox[2], bbox[3]], [bbox[0], bbox[3]],
                                                          [bbox[0], bbox[1]]]]},
    datetime="2022-01-01T00:00:00Z/2022-12-31T23:59:59Z",
    collections=["itslive-granules"],
    limit=100,  # items per HTTP request
)

total = 0
for page in search.pages_as_dicts():
    features = page.get("features", [])
    total += len(features)
    print(f"page: {len(features)} items | numberMatched={page.get('numberMatched')} | "
          f"next={any(l.get('rel')=='next' for l in page.get('links', []))}")
print(f"total: {total} items")

### Manual pagination, first N hrefs only

Stop early without fetching every page by breaking out of the loop:

In [ ]:
total = 0
for page in search.pages_as_dicts():
    for feat in page.get("features", []):
        href = feat["assets"]["data"]["href"]
        total += 1
        if total <= 3:
            print(href)
    if total >= 100:
        print("...stopping early after", total, "items")
        break

## 6. Manual pagination over rustac results

The serverless backends stream results as they are computed. `stream=True` returns a generator you can consume in chunks, e.g. to download or process batches:

In [ ]:
import itertools

gen = search(bbox=bbox, start="2022-01-01", end="2022-12-31",
              type="serverless", engine="rustac", stream=True)

batch = list(itertools.islice(gen, 3))
print(f"first batch of {len(batch)}:")
for href in batch:
    print(" ", href)

## 7. Legacy API

`itslive.velocity_pairs.find(...)` / `find_streaming(...)` remain as thin wrappers over the unified `search()`, mapping the legacy `engine="stac"` value to `type="pgstac"` and `engine="duckdb"/"rustac"` to `type="serverless"`.

In [ ]:
from itslive import velocity_pairs

urls = velocity_pairs.find(bbox=bbox, start="2022-01-01", end="2022-06-30",
                            engine="duckdb")
print(f"legacy find(): {len(urls)} velocity pairs")